# Price Tag Detection + Product Association

**Computer Vision Final Project**  
**Theme:** Price Tag Detection + Association  
**Task:** Detect price labels and match them to nearby products  
**Author / Team:** team (Mikita Malafei)

## Goal

This notebook implements a complete classical Computer Vision system for supermarket shelf images.

The system takes an input image and follows the required processing pipeline:

1. **Enhance** - improve image quality.
2. **Segment** - extract price-label candidate regions.
3. **Clean** - remove noise and artifacts from the binary mask.
4. **Detect** - localize price tags using mask geometry and row consistency.
5. **Decide / Associate** - match each detected price tag with the product region above it and produce an automatic final decision.

The final output contains stage-by-stage visualizations, detected price tags, product associations, a summary table, and saved result files.

# Project Approach

The project uses classical OpenCV methods instead of deep learning. This keeps the pipeline explainable and directly aligned with the course requirements.

The key observation is shelf geometry:

> Price tags are usually small rectangular labels arranged in horizontal rows on shelf strips, and the related product is usually located above the tag.

Therefore, the detector focuses on compact rectangular components in the cleaned segmentation mask and then validates them using color, text-like structure, row alignment, and spacing between neighboring tags.

In [ ]:
!pip -q install opencv-python matplotlib numpy pandas

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from google.colab import files
from pathlib import Path

plt.rcParams["figure.dpi"] = 120

# Upload Test Image

Upload one supermarket shelf image. The image should contain visible price labels and products.

The notebook is designed for frontal or near-frontal shelf photos. Strong perspective distortion, motion blur, and unusual label colors may reduce detection quality.

In [ ]:
uploaded = files.upload()

if not uploaded:
    raise ValueError("Please upload at least one image file.")

filename = list(uploaded.keys())[0]

image_bgr = cv2.imread(filename)
if image_bgr is None:
    raise ValueError("Image could not be loaded. Please upload a valid image file.")

image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
print("Loaded image:", filename)
print("Original size:", image.shape[1], "x", image.shape[0])

In [ ]:
def resize_for_processing(img, max_width=1400):
    """Resize only very large images to keep Colab processing fast.
    Normal classroom/demo images are kept unchanged.
    """
    h, w = img.shape[:2]
    if w <= max_width:
        return img.copy(), 1.0

    scale = max_width / float(w)
    resized = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    return resized, scale

image, scale = resize_for_processing(image)
print("Processing size:", image.shape[1], "x", image.shape[0])
print("Scale factor:", round(scale, 3))

In [ ]:
def show_image(img, title="", gray=False, figsize=(12, 7)):
    plt.figure(figsize=figsize)
    if gray:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()

show_image(image, "Original Image")

# Stage 1 - Enhance

The enhancement stage improves image quality before segmentation.

Methods used:

- light Gaussian Blur for noise reduction;
- CLAHE for local contrast enhancement.

Detection still uses a sharp copy of the input image, because price labels contain small text and thin borders that should not be blurred too much.

In [ ]:
def enhance_image(img_rgb):
    # Keep one sharp copy for detection. Blur is useful for visualization/noise reduction,
    # but it can destroy tiny price digits and thin label borders.
    blurred = cv2.GaussianBlur(img_rgb, (3, 3), 0)

    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=1.8, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l)

    enhanced_lab = cv2.merge((l_enhanced, a, b))
    enhanced_rgb = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2RGB)
    return blurred, enhanced_rgb

blurred, enhanced = enhance_image(image)
detection_image = image.copy()

fig, ax = plt.subplots(1, 3, figsize=(20, 7))
ax[0].imshow(image)
ax[0].set_title("Original / Detection Input")
ax[1].imshow(blurred)
ax[1].set_title("Light Gaussian Blur")
ax[2].imshow(enhanced)
ax[2].set_title("CLAHE Enhanced")
for a in ax:
    a.axis("off")
plt.show()

# Stage 2 - Segment Price Tag Candidates

The segmentation stage extracts candidate price-label pixels in HSV color space.

The detector focuses on common price-label colors:

- red sale strips;
- yellow price areas;
- white or light label paper.

If the image is too bright and the white mask floods too much of the scene, the notebook automatically switches to a safer segmentation mode that relies more on red/yellow seed regions.

In [ ]:
def segment_price_tag_pixels(img_rgb):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)

    # Core price-label colors. Cyan/pink are intentionally excluded because many products share them.
    red1 = cv2.inRange(hsv, np.array([0, 55, 80]), np.array([13, 255, 255]))
    red2 = cv2.inRange(hsv, np.array([166, 55, 80]), np.array([180, 255, 255]))
    red = cv2.bitwise_or(red1, red2)

    yellow = cv2.inRange(hsv, np.array([14, 45, 95]), np.array([45, 255, 255]))
    white = cv2.inRange(hsv, np.array([0, 0, 145]), np.array([180, 95, 255]))

    normal_mask = cv2.bitwise_or(cv2.bitwise_or(red, yellow), white)

    white_ratio = np.count_nonzero(white) / float(white.size)
    normal_ratio = np.count_nonzero(normal_mask) / float(normal_mask.size)

    if white_ratio > 0.48 or normal_ratio > 0.58:
        # Bright/flooded image mode: use red/yellow labels as seeds and grow them moderately.
        seeds = cv2.bitwise_or(red, yellow)
        seed_ratio = np.count_nonzero(seeds) / float(seeds.size)

        if seed_ratio < 0.002:
            # Fallback for images with mostly white labels and weak red/yellow areas.
            strict_white = cv2.inRange(hsv, np.array([0, 0, 185]), np.array([180, 55, 255]))
            seeds = cv2.bitwise_or(seeds, strict_white)

        mask = cv2.dilate(
            seeds,
            cv2.getStructuringElement(cv2.MORPH_RECT, (13, 5)),
            iterations=1
        )
        mode = "adaptive_flooded_image"
    else:
        mask = normal_mask
        mode = "normal"

    return hsv, mask, white, yellow, red, mode, white_ratio, normal_ratio

hsv, raw_mask, white_mask, yellow_mask, red_mask, segmentation_mode, white_ratio, normal_ratio = segment_price_tag_pixels(detection_image)

print("Segmentation mode:", segmentation_mode)
print("White mask ratio:", round(white_ratio, 3))
print("Combined mask ratio:", round(normal_ratio, 3))

fig, ax = plt.subplots(1, 4, figsize=(22, 6))
ax[0].imshow(white_mask, cmap="gray")
ax[0].set_title("White / Light")
ax[1].imshow(yellow_mask, cmap="gray")
ax[1].set_title("Yellow")
ax[2].imshow(red_mask, cmap="gray")
ax[2].set_title("Red")
ax[3].imshow(raw_mask, cmap="gray")
ax[3].set_title("Final Candidate Mask")
for a in ax:
    a.axis("off")
plt.show()

# Stage 3 - Clean Mask

The raw segmentation mask contains noise and small artifacts. Morphological processing is used to create cleaner candidate regions before contour detection.

Methods used:

- morphological closing;
- morphological opening;
- connected component filtering.

In [ ]:
def clean_candidate_mask(mask):
    # Horizontal closing connects the colored strip, paper area, and digits of one price tag.
    close_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 3))
    open_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))

    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, close_kernel, iterations=1)
    opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, open_kernel, iterations=1)

    h, w = mask.shape[:2]
    min_area = max(12, int(0.000015 * h * w))

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(opened, connectivity=8)
    cleaned = np.zeros_like(mask)

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_area:
            cleaned[labels == i] = 255

    return closed, opened, cleaned

closed_mask, opened_mask, clean_mask = clean_candidate_mask(raw_mask)

fig, ax = plt.subplots(1, 4, figsize=(22, 6))
ax[0].imshow(raw_mask, cmap="gray")
ax[0].set_title("Raw Mask")
ax[1].imshow(closed_mask, cmap="gray")
ax[1].set_title("Closing")
ax[2].imshow(opened_mask, cmap="gray")
ax[2].set_title("Opening")
ax[3].imshow(clean_mask, cmap="gray")
ax[3].set_title("Clean Mask")
for a in ax:
    a.axis("off")
plt.show()

# Stage 4 - Detect Price Tags

The detector uses the cleaned binary mask as the main source of evidence.

Detection logic:

1. Find small candidate components on the clean mask.
2. Validate that each candidate is a compact filled rectangle.
3. Reject irregular, fragmented, or dark product-label regions.
4. Confirm candidates using label-like color and text evidence.
5. Group candidates into horizontal rows.
6. Keep only plausible rows with aligned rectangles of similar height and area.
7. Remove overlapping boxes with non-maximum suppression.

This approach is intentionally conservative: a price tag should look like a separated rectangular label on the binary mask, not just like any colorful product fragment.

In [ ]:
def box_area(box):
    x, y, w, h = box
    return max(0, w) * max(0, h)

def iou(box_a, box_b):
    ax, ay, aw, ah = box_a
    bx, by, bw, bh = box_b
    ax2, ay2 = ax + aw, ay + ah
    bx2, by2 = bx + bw, by + bh

    ix1, iy1 = max(ax, bx), max(ay, by)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = box_area(box_a) + box_area(box_b) - inter
    return inter / union if union else 0

def non_max_suppression(boxes, scores=None, iou_threshold=0.35):
    if not boxes:
        return []
    if scores is None:
        scores = [box_area(b) for b in boxes]

    order = np.argsort(scores)[::-1]
    selected = []

    while len(order) > 0:
        idx = int(order[0])
        current = boxes[idx]
        selected.append(current)

        remaining = []
        for j in order[1:]:
            if iou(current, boxes[int(j)]) < iou_threshold:
                remaining.append(j)
        order = np.array(remaining)

    return selected

In [ ]:
def center(box):
    x, y, w, h = box
    return (x + w // 2, y + h // 2)


# Mask-first detector preset.
# Main idea: price tags are small filled rectangles on the clean binary mask.
MIN_RECT_SCORE = 0.42
MIN_ROW_SIZE = 3


def label_color_features(img_rgb, box):
    x, y, w, h = box
    roi = img_rgb[y:y+h, x:x+w]
    if roi.size == 0:
        return 0.0, 0.0, 0.0

    hsv = cv2.cvtColor(roi, cv2.COLOR_RGB2HSV)
    hue = hsv[:, :, 0]
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]

    red = (((hue <= 13) | (hue >= 166)) & (sat > 45) & (val > 75))
    yellow = ((hue >= 14) & (hue <= 45) & (sat > 35) & (val > 85))
    white = ((sat < 110) & (val > 135))

    core = red | yellow | white
    warm = red | yellow
    return float(np.mean(core)), float(np.mean(warm)), float(np.mean(white))


def price_text_score(img_rgb, box):
    x, y, w, h = box
    roi = img_rgb[y:y+h, x:x+w]
    if roi.size == 0:
        return 0.0

    gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)
    dark = cv2.inRange(gray, 0, 130)

    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(dark, connectivity=8)
    digit_like = 0
    dark_area = 0

    for i in range(1, num_labels):
        sw = stats[i, cv2.CC_STAT_WIDTH]
        sh = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]
        dark_area += area

        if area < 2:
            continue
        if 1 <= sw <= max(5, int(0.60 * w)) and 2 <= sh <= max(5, int(0.95 * h)):
            digit_like += 1

    component_score = min(digit_like / 4.0, 1.0)
    ink_score = min((dark_area / float(max(1, w * h))) / 0.26, 1.0)
    return float(0.70 * component_score + 0.30 * ink_score)


def group_boxes_by_row(boxes, tolerance):
    rows = []
    for box in sorted(boxes, key=lambda b: center(b)[1]):
        _, cy = center(box)
        placed = False
        for row in rows:
            row_y = np.median([center(b)[1] for b in row])
            if abs(cy - row_y) <= tolerance:
                row.append(box)
                placed = True
                break
        if not placed:
            rows.append([box])
    return [sorted(row, key=lambda b: center(b)[0]) for row in rows]



def mask_rectangular_shape_score(mask, box):
    x, y, w, h = box
    roi = mask[y:y+h, x:x+w]
    if roi.size == 0:
        return 0.0, {}

    binary = roi > 0
    fill_ratio = np.mean(binary)

    row_fill = np.mean(binary, axis=1)
    col_fill = np.mean(binary, axis=0)

    # A real label rectangle has many rows/columns with substantial foreground coverage.
    strong_rows = np.mean(row_fill >= 0.55)
    strong_cols = np.mean(col_fill >= 0.35)
    median_row_fill = float(np.median(row_fill))
    median_col_fill = float(np.median(col_fill))

    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(roi, connectivity=8)
    if num_labels <= 1:
        largest_component_ratio = 0.0
    else:
        areas = stats[1:, cv2.CC_STAT_AREA]
        largest_component_ratio = float(np.max(areas) / max(1, np.count_nonzero(roi)))

    # Compare the mask component with an ideal filled rectangle.
    contour_mask = roi.copy()
    contours, _ = cv2.findContours(contour_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        contour_area = cv2.contourArea(largest)
        rectangularity = contour_area / float(max(1, w * h))
        hull = cv2.convexHull(largest)
        hull_area = cv2.contourArea(hull)
        solidity = contour_area / float(max(1, hull_area))
    else:
        rectangularity = 0.0
        solidity = 0.0

    score = (
        0.22 * min(fill_ratio / 0.62, 1.0) +
        0.20 * strong_rows +
        0.16 * strong_cols +
        0.16 * min(median_row_fill / 0.60, 1.0) +
        0.10 * min(median_col_fill / 0.42, 1.0) +
        0.10 * largest_component_ratio +
        0.06 * solidity
    )

    metrics = {
        "fill_ratio": float(fill_ratio),
        "strong_rows": float(strong_rows),
        "strong_cols": float(strong_cols),
        "median_row_fill": float(median_row_fill),
        "median_col_fill": float(median_col_fill),
        "largest_component_ratio": float(largest_component_ratio),
        "rectangularity": float(rectangularity),
        "solidity": float(solidity),
    }
    return float(score), metrics



def label_background_score(img_rgb, box):
    x, y, w, h = box
    roi = img_rgb[y:y+h, x:x+w]
    if roi.size == 0:
        return 0.0, {"bright_ratio": 0.0, "dark_ratio": 1.0, "warm_ratio": 0.0, "mean_value": 0.0}

    hsv = cv2.cvtColor(roi, cv2.COLOR_RGB2HSV)
    hue = hsv[:, :, 0]
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]

    bright = val > 135
    dark = val < 95
    warm = (((hue <= 13) | (hue >= 166)) & (sat > 45) & (val > 75)) | ((hue >= 14) & (hue <= 45) & (sat > 35) & (val > 85))
    light_paper = (sat < 115) & (val > 135)

    bright_ratio = float(np.mean(bright))
    dark_ratio = float(np.mean(dark))
    warm_ratio = float(np.mean(warm))
    paper_ratio = float(np.mean(light_paper))
    mean_value = float(np.mean(val))

    # True shelf labels usually have a visible bright/warm background.
    # Dark product labels with white text should score low here.
    score = (
        0.36 * min(bright_ratio / 0.48, 1.0) +
        0.30 * min((warm_ratio + paper_ratio) / 0.34, 1.0) +
        0.20 * min(mean_value / 165.0, 1.0) +
        0.14 * max(0.0, 1.0 - min(dark_ratio / 0.45, 1.0))
    )

    return float(score), {
        "bright_ratio": bright_ratio,
        "dark_ratio": dark_ratio,
        "warm_ratio": warm_ratio,
        "paper_ratio": paper_ratio,
        "mean_value": mean_value,
    }

def mask_rectangle_candidates(mask, img_rgb):
    h_img, w_img = mask.shape[:2]
    image_area = h_img * w_img

    # Use a very light close only to connect parts of the same label for candidate proposal.
    # Shape validation below is done on the original clean_mask, not on this closed version.
    proposal_mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (5, 2)),
        iterations=1
    )

    contours, _ = cv2.findContours(proposal_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    candidates = []
    scores = {}

    min_area = max(22, int(0.000040 * image_area))
    max_area = int(0.014 * image_area)
    min_w = max(11, int(0.012 * w_img))
    max_w = int(0.24 * w_img)
    min_h = max(6, int(0.007 * h_img))
    max_h = int(0.075 * h_img)

    for contour in contours:
        proposal_area = cv2.contourArea(contour)
        if proposal_area < min_area or proposal_area > max_area:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        if w < min_w or w > max_w or h < min_h or h > max_h:
            continue

        aspect = w / float(h)
        if aspect < 1.15 or aspect > 8.5:
            continue

        box = (x, y, w, h)
        shape_score, shape = mask_rectangular_shape_score(mask, box)

        # Strict mask-first test: no filled rectangle on the clean mask means no price tag.
        if shape_score < 0.48:
            continue
        if shape["fill_ratio"] < 0.42:
            continue
        if shape["strong_rows"] < 0.42:
            continue
        if shape["median_row_fill"] < 0.38:
            continue
        if shape["largest_component_ratio"] < 0.58:
            continue

        color_score, warm_score, white_score = label_color_features(img_rgb, box)
        text_score = price_text_score(img_rgb, box)
        background_score, background = label_background_score(img_rgb, box)

        # Color/text is secondary confirmation, not the main detector.
        # Dark product labels with white text are rejected here.
        if color_score < 0.16:
            continue
        if background_score < 0.42:
            continue
        if background["dark_ratio"] > 0.52 and warm_score < 0.10:
            continue
        if background["bright_ratio"] < 0.28 and warm_score < 0.12:
            continue
        if warm_score < 0.035 and white_score < 0.20 and text_score < 0.18:
            continue

        aspect_score = 1.0 - min(abs(aspect - 2.8) / 5.5, 1.0)
        rect_score = min(shape["rectangularity"] / 0.55, 1.0)

        score = (
            0.42 * shape_score +
            0.18 * background_score +
            0.14 * rect_score +
            0.12 * color_score +
            0.08 * text_score +
            0.06 * aspect_score
        )

        if score < MIN_RECT_SCORE:
            continue

        candidates.append(box)
        scores[box] = score

    return candidates, scores

def gap_is_clean(mask, left_box, right_box, row_y, row_h):
    lx, ly, lw, lh = left_box
    rx, ry, rw, rh = right_box
    x1 = lx + lw
    x2 = rx
    if x2 <= x1:
        return True

    h_img, w_img = mask.shape[:2]
    strip_h = max(4, int(0.65 * row_h))
    y1 = max(0, int(row_y - strip_h / 2))
    y2 = min(h_img, int(row_y + strip_h / 2))
    gap = mask[y1:y2, x1:x2]
    if gap.size == 0:
        return True

    # Between neighboring tags on the clean mask there should be mostly black background.
    white_ratio = np.count_nonzero(gap) / float(gap.size)
    return white_ratio < 0.28


def row_is_plausible(row_boxes, scores, img_width):
    if not row_boxes:
        return False

    row_boxes = sorted(row_boxes, key=lambda b: center(b)[0])
    count = len(row_boxes)
    x_left = min(b[0] for b in row_boxes)
    x_right = max(b[0] + b[2] for b in row_boxes)
    span = (x_right - x_left) / float(max(1, img_width))
    avg_score = float(np.mean([scores.get(b, 0.0) for b in row_boxes]))

    # A real price-label row usually has several separated rectangles across the shelf rail.
    if count >= 3 and span >= 0.18:
        return True

    # Allow two-tag rows only if they are very strong and not just two nearby product fragments.
    if count == 2 and span >= 0.30 and avg_score >= 0.62:
        return True

    return False


def recompute_row_centers_from_boxes(boxes, img_h):
    if not boxes:
        return []
    rows = group_boxes_by_row(boxes, max(10, int(0.026 * img_h)))
    centers = []
    for row in rows:
        if len(row) >= 2:
            centers.append(float(np.median([center(b)[1] for b in row])))
    return sorted(centers)


def row_consensus_filter(mask, boxes, scores):
    if not boxes:
        return [], []

    h_img, w_img = mask.shape[:2]
    rows = group_boxes_by_row(boxes, max(10, int(0.026 * h_img)))
    kept = []

    for row in rows:
        if len(row) < 2:
            continue

        row_y = float(np.median([center(b)[1] for b in row]))
        heights = np.array([b[3] for b in row], dtype=float)
        areas = np.array([b[2] * b[3] for b in row], dtype=float)
        med_h = float(np.median(heights))
        med_area = float(np.median(areas))

        aligned = []
        for box in row:
            x, y, w, h = box
            area = w * h
            cy = center(box)[1]

            y_ok = abs(cy - row_y) <= max(5, 0.45 * med_h)
            h_ok = 0.62 * med_h <= h <= 1.62 * med_h
            area_ok = 0.42 * med_area <= area <= 2.75 * med_area

            if y_ok and h_ok and area_ok:
                aligned.append(box)

        if len(aligned) < 2:
            continue

        aligned = sorted(aligned, key=lambda b: center(b)[0])

        # Verify that neighboring rectangles are separated by black gaps on the mask.
        gap_votes = 0
        for left, right in zip(aligned[:-1], aligned[1:]):
            if gap_is_clean(mask, left, right, row_y, med_h):
                gap_votes += 1

        if len(aligned) >= 3 and gap_votes < len(aligned) - 1:
            continue
        if len(aligned) == 2 and gap_votes < 1:
            continue

        row_kept = [b for b in aligned if scores.get(b, 0.0) >= max(0.38, MIN_RECT_SCORE - 0.03)]

        if not row_is_plausible(row_kept, scores, w_img):
            continue

        kept.extend(row_kept)

    kept_scores = [scores[b] for b in kept]
    kept = non_max_suppression(kept, scores=kept_scores, iou_threshold=0.20)
    kept = sorted(kept, key=lambda b: (b[1], b[0]))

    # Recompute debug row lines after NMS so false/deleted rows do not remain visible.
    row_centers = recompute_row_centers_from_boxes(kept, h_img)
    return kept, row_centers

def detect_price_tags(mask, img_rgb):
    candidates, scores = mask_rectangle_candidates(mask, img_rgb)
    boxes, row_centers = row_consensus_filter(mask, candidates, scores)

    print("Detection stage completed.")
    print("Candidate rectangles:", len(candidates))
    print("Valid tag rows:", len(row_centers), [round(y, 1) for y in row_centers])
    return boxes, row_centers

price_boxes, tag_row_centers = detect_price_tags(clean_mask, detection_image)
print("Detected price tags:", len(price_boxes))

row_vis = detection_image.copy()
for y in tag_row_centers:
    cv2.line(row_vis, (0, int(round(y))), (row_vis.shape[1], int(round(y))), (255, 255, 0), 2)
show_image(row_vis, "Detected Price Tag Rows")

In [ ]:
def draw_numbered_boxes(img, boxes, color=(0, 255, 0), prefix=""):
    out = img.copy()
    for idx, (x, y, w, h) in enumerate(boxes, start=1):
        cv2.rectangle(out, (x, y), (x + w, y + h), color, 2)
        label = f"{prefix}{idx}"
        cv2.putText(out, label, (x, max(15, y - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return out

price_vis = draw_numbered_boxes(detection_image, price_boxes, color=(0, 255, 0), prefix="T")
show_image(price_vis, "Detected Price Tags")

# Stage 5 - Product Association and Decision

For each detected price tag, the system creates a product candidate region above the tag.

The association is based on supermarket shelf layout:

- price labels are located on shelf strips;
- the corresponding product is usually above the label;
- neighboring tags define approximate horizontal product regions.

The final decision reports how many detected labels have confident product-region associations.

In [ ]:
def clamp_box(x1, y1, x2, y2, img_shape):
    h, w = img_shape[:2]
    x1 = int(max(0, min(w - 1, x1)))
    y1 = int(max(0, min(h - 1, y1)))
    x2 = int(max(0, min(w, x2)))
    y2 = int(max(0, min(h, y2)))
    if x2 <= x1 or y2 <= y1:
        return None
    return (x1, y1, x2 - x1, y2 - y1)


def center(box):
    x, y, w, h = box
    return (x + w // 2, y + h // 2)


def build_tag_rows(tag_boxes, img_h):
    tolerance = max(16, int(0.03 * img_h))
    rows = group_boxes_by_row(tag_boxes, tolerance)
    return [sorted(row, key=lambda b: center(b)[0]) for row in rows]


def product_region_for_tag(tag_box, row_boxes, previous_row_y, img_shape):
    img_h, img_w = img_shape[:2]
    x, y, w, h = tag_box
    cx, cy = center(tag_box)

    centers = [center(b)[0] for b in row_boxes]
    idx = row_boxes.index(tag_box)

    if idx > 0:
        left = int((centers[idx - 1] + cx) / 2)
    else:
        left = cx - max(int(1.45 * w), int(0.045 * img_w))

    if idx < len(row_boxes) - 1:
        right = int((cx + centers[idx + 1]) / 2)
    else:
        right = cx + max(int(1.45 * w), int(0.045 * img_w))

    margin = max(6, int(0.12 * (right - left)))
    x1 = left - margin
    x2 = right + margin

    y2 = y - max(4, int(0.20 * h))
    if previous_row_y is None:
        y1 = y2 - max(int(3.6 * h), int(0.13 * img_h))
    else:
        y1 = int(previous_row_y + max(10, 0.8 * h))

    max_region_h = int(0.22 * img_h)
    if y2 - y1 > max_region_h:
        y1 = y2 - max_region_h

    min_region_h = max(int(2.2 * h), 24)
    if y2 - y1 < min_region_h:
        y1 = y2 - min_region_h

    return clamp_box(x1, y1, x2, y2, img_shape)


def association_confidence(img_rgb, region_box):
    if region_box is None:
        return 0.0, {"edge_density": 0.0, "texture": 0.0}

    x, y, w, h = region_box
    roi = img_rgb[y:y+h, x:x+w]
    if roi.size == 0:
        return 0.0, {"edge_density": 0.0, "texture": 0.0}

    gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    edge_density = np.count_nonzero(edges) / float(edges.size)
    texture = np.std(gray) / 64.0
    non_empty = np.mean(gray > 35)

    score = 0.55 * min(edge_density / 0.12, 1.0) + 0.35 * min(texture, 1.0) + 0.10 * non_empty
    return float(score), {"edge_density": float(edge_density), "texture": float(texture)}


def associate_tags_to_products(img_rgb, tag_boxes, confidence_threshold=0.34):
    associations = []
    rows = build_tag_rows(tag_boxes, img_rgb.shape[0])

    for row_idx, row in enumerate(rows):
        previous_row_y = None
        if row_idx > 0:
            previous_row_y = np.mean([center(b)[1] for b in rows[row_idx - 1]])

        for tag in row:
            product_box = product_region_for_tag(tag, row, previous_row_y, img_rgb.shape)
            confidence, metrics = association_confidence(img_rgb, product_box)
            status = "ASSOCIATED" if confidence >= confidence_threshold else "WEAK_ASSOCIATION"

            associations.append({
                "id": len(associations) + 1,
                "tag_box": tag,
                "product_box": product_box,
                "confidence": confidence,
                "status": status,
                **metrics,
            })

    return associations

associations = associate_tags_to_products(detection_image, price_boxes)
print("Association stage completed.")
print("Associations created:", len(associations))

In [ ]:
def draw_associations(img_rgb, associations, draw_weak_regions=False):
    out = img_rgb.copy()

    for item in associations:
        tag = item["tag_box"]
        product = item["product_box"]
        idx = item["id"]
        confidence = item["confidence"]
        status = item["status"]

        tx, ty, tw, th = tag
        tag_color = (0, 255, 0) if status == "ASSOCIATED" else (255, 220, 0)
        product_color = (255, 90, 0)
        line_color = (255, 0, 0)

        cv2.rectangle(out, (tx, ty), (tx + tw, ty + th), tag_color, 2)
        cv2.putText(out, f"T{idx}", (tx, max(15, ty - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.48, tag_color, 2)

        if product is not None and (status == "ASSOCIATED" or draw_weak_regions):
            px, py, pw, ph = product
            cv2.rectangle(out, (px, py), (px + pw, py + ph), product_color, 2)
            cv2.putText(out, f"P{idx} {confidence:.2f}", (px, max(15, py - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, product_color, 2)

            if status == "ASSOCIATED":
                pc = center(product)
                tc = center(tag)
                cv2.line(out, pc, tc, line_color, 2)
                cv2.circle(out, pc, 3, product_color, -1)
                cv2.circle(out, tc, 3, tag_color, -1)

    return out

association_result = draw_associations(detection_image, associations)
show_image(association_result, "Price Tag Detection + Product Association", figsize=(14, 9))

# Decision Summary

The final decision is produced for every detected price tag:

- `ASSOCIATED`: the system found a visually meaningful product region above the price tag.
- `WEAK_ASSOCIATION`: the region above the tag is weak or unclear, so manual verification may be needed.

This directly answers the project topic: detected price labels are matched to nearby product regions.

In [ ]:
summary_rows = []
for item in associations:
    tag = item["tag_box"]
    product = item["product_box"]
    summary_rows.append({
        "ID": item["id"],
        "Tag box": tag,
        "Product region": product,
        "Confidence": round(item["confidence"], 3),
        "Edge density": round(item["edge_density"], 3),
        "Texture": round(item["texture"], 3),
        "Status": item["status"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
total_tags = len(associations)
associated = sum(1 for item in associations if item["status"] == "ASSOCIATED")
weak = total_tags - associated

if total_tags == 0:
    final_status = "NO PRICE TAGS DETECTED"
    decision_message = "The system did not detect valid price tags in the image."
elif weak == 0:
    final_status = "OK"
    decision_message = "All detected price tags have associated product regions."
else:
    final_status = "REVIEW NEEDED"
    decision_message = "Some detected price tags have weak product-region associations."

print("Final decision")
print("Detected price tags:", total_tags)
print("Associated product regions:", associated)
print("Weak associations:", weak)
print("STATUS:", final_status)
print(decision_message)

# Stage Overview

The following overview summarizes the full pipeline in one figure. This image is useful for the README and video demonstration.

In [ ]:
def show_stage_overview():
    fig, ax = plt.subplots(2, 3, figsize=(18, 10))

    ax[0, 0].imshow(image)
    ax[0, 0].set_title("1. Original")

    ax[0, 1].imshow(enhanced)
    ax[0, 1].set_title("2. Enhanced")

    ax[0, 2].imshow(raw_mask, cmap="gray")
    ax[0, 2].set_title("3. Segmentation Mask")

    ax[1, 0].imshow(clean_mask, cmap="gray")
    ax[1, 0].set_title("4. Clean Mask")

    ax[1, 1].imshow(price_vis)
    ax[1, 1].set_title("5. Detection")

    ax[1, 2].imshow(association_result)
    ax[1, 2].set_title("6. Association + Decision")

    for a in ax.ravel():
        a.axis("off")

    plt.tight_layout()
    plt.show()

show_stage_overview()

# Save Result Files

All required output files are saved into `cv_project_outputs`:

- original image;
- enhanced image;
- segmentation mask;
- cleaned mask;
- detection result;
- final association result;
- association summary table.

In [ ]:
output_dir = Path("cv_project_outputs")
output_dir.mkdir(exist_ok=True)

cv2.imwrite(str(output_dir / "01_original.png"), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
cv2.imwrite(str(output_dir / "02_enhanced.png"), cv2.cvtColor(enhanced, cv2.COLOR_RGB2BGR))
cv2.imwrite(str(output_dir / "03_segmentation_mask.png"), raw_mask)
cv2.imwrite(str(output_dir / "04_clean_mask.png"), clean_mask)
cv2.imwrite(str(output_dir / "05_detected_price_tags.png"), cv2.cvtColor(price_vis, cv2.COLOR_RGB2BGR))
cv2.imwrite(str(output_dir / "06_association_result.png"), cv2.cvtColor(association_result, cv2.COLOR_RGB2BGR))

summary_df.to_csv(output_dir / "association_summary.csv", index=False)

with open(output_dir / "final_decision.txt", "w", encoding="utf-8") as f:
    f.write(f"Detected price tags: {total_tags}\n")
    f.write(f"Associated product regions: {associated}\n")
    f.write(f"Weak associations: {weak}\n")
    f.write(f"Status: {final_status}\n")
    f.write(decision_message + "\n")

print("Saved outputs to:", output_dir)
for path in sorted(output_dir.iterdir()):
    print("-", path)

# Conclusion

This notebook implements a complete classical Computer Vision pipeline for **Price Tag Detection + Product Association**.

The system satisfies the required course pipeline:

- **Enhance:** Gaussian Blur + CLAHE;
- **Segment:** HSV thresholding with adaptive fallback for bright images;
- **Clean:** morphology + connected component filtering;
- **Detect:** contour-based rectangular price-tag detection from the clean mask;
- **Decide:** automatic product-region association and final status output.

The method works best on frontal shelf images where price labels are visible and arranged in clear rows. Performance may decrease with strong perspective distortion, heavy occlusion, unusual label colors, or very low image quality.